# Government Services RAG Pipeline — Granite Switch Intrinsics

> For softer start, you can begin with
> [`hello_mellea.ipynb`](../quickstart/hello_mellea.ipynb) — it explains the prerequisites
> (composed model, vLLM server, mellea install) and walks through each intrinsic in isolation.

This notebook demonstrates a conversational RAG pipeline where **all AI capabilities run through a single vLLM endpoint**. This tutorial uses the vLLM backend because the mellea intrinsics API (guardian, RAG, citations) currently supports vLLM only. Guardian, query rewrite, clarification, and citations are embedded adapters baked into the Granite Switch model checkpoint and activated via control tokens — no separate model services, no dynamic LoRA loading at inference time.

**Pipeline steps:**
1. **Guardian** — blocks harmful content first, then out-of-scope queries (not about government services)
2. **Query Rewrite** — disambiguates the query using conversation history
3. **ChromaDB Retrieval** — dense vector search with `granite-embedding-small-english-r2`
4. **Answerability** — exits early if the retrieved documents cannot answer the query
5. **Query Clarification** — asks a follow-up if the retrieved context isn't sufficient
6. **Base Model** — generates the answer grounded in retrieved documents
7. **Citations** — extracts the document spans that support the answer

## 1 · Configuration

In [ ]:
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    load_dotenv(Path("../.env"), override=False)
except ImportError:
    pass

# ── vLLM server ───────────────────────────────────────────────────────────────
# URL of the running vLLM OpenAI-compatible endpoint.
VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", "http://localhost:8000/v1")

# Model name as reported by GET /v1/models (usually the path/repo used at launch).
VLLM_MODEL_NAME = os.environ.get("VLLM_MODEL_NAME", "ibm-granite/granite-switch-4.1-3b-preview")

# HF Hub repo ID (or local path) to load I/O configs for the embedded adapters.
GRANITE_SWITCH_SOURCE = os.environ.get("GRANITE_SWITCH_SOURCE", VLLM_MODEL_NAME)

# Guardian: which safety criterion to evaluate
GUARDIAN_CRITERIA = "harm"  # harm | social_bias | groundedness | jailbreak | ...

# ── Embedding model (used to build + query ChromaDB) ─────────────────────────
EMBEDDING_MODEL_ID = "ibm-granite/granite-embedding-small-english-r2"

# ── ChromaDB persistence path ─────────────────────────────────────────────────
# Share this directory (zipped) to skip the extraction step entirely.
CHROMA_PATH = "./govt_chroma"

# ── Corpus source (only needed when building the index from scratch) ─────────
# govt.jsonl: 49k government-service passages from IBM mt-rag-benchmark.
GOVT_JSONL_URL  = "https://github.com/IBM/mt-rag-benchmark/raw/main/corpora/passage_level/govt.jsonl.zip"
GOVT_JSONL_PATH = "./govt.jsonl"

# ── Retrieval ─────────────────────────────────────────────────────────────────
TOP_K = 20

# ── Answerability ─────────────────────────────────────────────────────────────
ANSWERABILITY_THRESHOLD = 0.5

print(f"vLLM:      {VLLM_BASE_URL}  ({VLLM_MODEL_NAME})")
print(f"Embedding: {EMBEDDING_MODEL_ID}")
print(f"ChromaDB:  {CHROMA_PATH}")

## 2 · Build or load vector corpus (ChromaDB + Granite embeddings)

**Two paths — runs the right one automatically:**

- **`./govt_chroma` exists** → loads the persisted index instantly.
- **Directory absent** → downloads `govt.jsonl.zip` from
  [IBM mt-rag-benchmark](https://github.com/IBM/mt-rag-benchmark) (49k passages),
  embeds with `ibm-granite/granite-embedding-small-english-r2`, and saves to `./govt_chroma`.

To share the pre-built index: zip `./govt_chroma/` and distribute it.
Recipients unzip it next to the notebook and this cell becomes a fast load.

In [ ]:
import io, json, os, time, zipfile
import httpx
import torch
import chromadb
from chromadb import EmbeddingFunction, Documents, Embeddings
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm


class GraniteEmbeddingFunction(EmbeddingFunction):
    """ChromaDB EmbeddingFunction backed by ibm-granite/granite-embedding-*-r2."""

    def __init__(self, model_id=EMBEDDING_MODEL_ID, batch_size=64):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self._device    = device
        self._batch     = batch_size
        self._tokenizer = AutoTokenizer.from_pretrained(model_id)
        self._model     = AutoModel.from_pretrained(model_id).to(device).eval()
        print(f"Granite embedding model ready on {device}  ({model_id})")
        if device=="cpu":
            print("Notice: running Embedding & indexing step on a cpu might take a very long time.")

    def __call__(self, input: Documents) -> Embeddings:
        all_embs = []
        for i in range(0, len(input), self._batch):
            batch = list(input[i : i + self._batch])
            enc   = self._tokenizer(
                batch, return_tensors="pt", truncation=True, max_length=512, padding=True
            )
            enc = {k: v.to(self._device) for k, v in enc.items()}
            with torch.no_grad():
                out = self._model(**enc)
            mask = enc["attention_mask"].unsqueeze(-1).float()
            emb  = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            all_embs.extend(emb.cpu().float().tolist())
        return all_embs


# EmbeddingFunction is always needed — for queries even when loading from disk.
granite_ef    = GraniteEmbeddingFunction()
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
chroma_collection = chroma_client.get_or_create_collection(
    name="govt",
    embedding_function=granite_ef,
    metadata={"hnsw:space": "cosine"},
)

if chroma_collection.count() > 0:
    print(f"Loaded from {CHROMA_PATH}  ({chroma_collection.count():,} docs).")
else:
    # Download govt.jsonl.zip if the raw jsonl is not already on disk.
    if not os.path.exists(GOVT_JSONL_PATH):
        print(f"Downloading {GOVT_JSONL_URL} …")
        t0 = time.time()
        with httpx.Client(follow_redirects=True, timeout=120.0) as client:
            resp = client.get(GOVT_JSONL_URL)
            resp.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            inner = [n for n in zf.namelist() if n.endswith(".jsonl")][0]
            with zf.open(inner) as src, open(GOVT_JSONL_PATH, "wb") as dst:
                dst.write(src.read())
        print(f"Saved {GOVT_JSONL_PATH} in {time.time() - t0:.1f}s.")

    print(f"Reading {GOVT_JSONL_PATH} → {CHROMA_PATH}…")
    t0 = time.time()
    ids, texts, metas = [], [], []
    with open(GOVT_JSONL_PATH) as f:
        for line in f:
            doc  = json.loads(line)
            text = doc.get("text", "").strip()
            if not text:
                continue
            ids.append(doc.get("_id", doc.get("id", str(len(ids)))))
            texts.append(text)
            metas.append({"title": doc.get("title", ""), "url": doc.get("url", "")})
    print(f"Read {len(ids):,} docs in {time.time() - t0:.1f}s.  Embedding & indexing…")

    t1 = time.time()
    BATCH = 500
    for i in tqdm(range(0, len(ids), BATCH), unit="batch", desc="indexing"):
        chroma_collection.upsert(
            ids       = ids  [i : i + BATCH],
            documents = texts[i : i + BATCH],
            metadatas = metas[i : i + BATCH],
        )
    print(f"Done. {chroma_collection.count():,} docs saved to {CHROMA_PATH} in {time.time() - t1:.1f}s.")


## 3 · Connect to vLLM backend
Registers the Granite Switch embedded adapters from `GRANITE_SWITCH_SOURCE`
so all intrinsics (guardian, RAG, citations) route through the correct control tokens.

In [ ]:
from IPython.display import display, Markdown

from mellea.backends.openai import OpenAIBackend
from mellea.stdlib.context import ChatContext
from mellea.stdlib.components import Document as MelleaDocument
from mellea.stdlib.components.chat import Message as MelleaMessage
import mellea.stdlib.functional as mfuncs

backend = OpenAIBackend(
    model_id=VLLM_MODEL_NAME,
    base_url=VLLM_BASE_URL,
    api_key="unused",
)
backend.register_embedded_adapter_model(GRANITE_SWITCH_SOURCE)
print(f"Backend ready — {backend.list_adapters()}")

## 4 · Functions + conversation state
Run this cell once after loading the backend.

In [ ]:
import json
from mellea.backends import ModelOption
from mellea.stdlib.components.intrinsic import rag
from mellea.stdlib.components.intrinsic.guardian import guardian_check

# ── Guardian criteria ─────────────────────────────────────────────────────────
# Full-conversation harm check.
GUARDIAN_HARM_CRITERIA = (
    "The conversation as a whole — including all user messages, assistant "
    "responses, and any referenced documents — is deemed harmful if it "
    "facilitates, encourages, or guides actions that pose harm across all "
    "contexts and audiences. Evaluate the cumulative intent of the full "
    "exchange, not just the latest message in isolation."
)

# Scope check: positive definition — message should be about government services.
# NOTE: threshold logic is FLIPPED here — a low guardian score means the message
# does NOT match this category, so we block on `score < 0.5`.
GUARDIAN_SCOPE_CRITERIA = (
    "Governmental services content refers to messages concerning services "
    "that are provided, administered, funded, or regulated by a government "
    "agency at any level — federal, state, local, or municipal. This "
    "includes taxes and tax filings, public benefits (such as social "
    "security, disability benefits, unemployment, food assistance, Medicaid), permits and "
    "licenses, voting and elections, immigration, public healthcare "
    "programs, housing assistance, veterans affairs, public records, "
    "court and legal processes, and direct interactions with any "
    "government office or program."
)

# Late instruction appended to the user message at generation time.
# Placed after history + documents to preserve KV cache prefix sharing.
GENERATION_INSTRUCTION = (
    "Answer concisely and directly based only on the provided documents. "
    "Do not repeat the question or add unnecessary preamble."
)

# ── Helpers ───────────────────────────────────────────────────────────────────
def build_context(history):
    """Build a ChatContext from conversation history.

    History entries may include a 'documents' key with retrieved doc texts.
    These are attached to the message so downstream intrinsics (including
    guardian) can see the full grounding context.
    """
    ctx = ChatContext()
    for m in history:
        docs = None
        if m.get("documents"):
            docs = [MelleaDocument(doc_id=str(i), text=t)
                    for i, t in enumerate(m["documents"])]
        ctx = ctx.add(MelleaMessage(m["role"], m["content"], documents=docs))
    return ctx

def to_mellea_docs(texts):
    return [MelleaDocument(doc_id=str(i), text=t) for i, t in enumerate(texts)]

# ── ChromaDB retrieval ────────────────────────────────────────────────────────
def retrieve_documents(query, top_k=TOP_K):
    results = chroma_collection.query(query_texts=[query], n_results=top_k)
    return results["documents"][0]

# ── Pipeline steps ────────────────────────────────────────────────────────────
def run_guardian_scope(query, history):
    """Check whether the query is within the government services domain."""
    ctx = build_context(history).add(MelleaMessage("user", query))
    return guardian_check(ctx, backend, GUARDIAN_SCOPE_CRITERIA, target_role="user")

def run_guardian_harm(query, history):
    """Check the full conversation for harmful intent."""
    ctx = build_context(history).add(MelleaMessage("user", query))
    return guardian_check(ctx, backend, GUARDIAN_HARM_CRITERIA, target_role="user")

def run_query_rewrite(query, history):
    return rag.rewrite_question(query, build_context(history), backend)

def run_answerability(query, history, documents):
    return rag.check_answerability(query, to_mellea_docs(documents), build_context(history), backend)

def run_query_clarification(query, history, documents):
    return rag.clarify_query(query, to_mellea_docs(documents), build_context(history), backend)

def _is_clear(clarification):
    """QC returns 'CLEAR' when no clarification is needed; accept prefix variants like 'CLEARLY'."""
    return clarification.strip().upper().startswith("CLEAR")

def run_base_model(query, history, documents):
    ctx = build_context(history)
    prompted_query = query + "\n\n" + GENERATION_INSTRUCTION
    out, _ = mfuncs.act(
        MelleaMessage("user", prompted_query, documents=to_mellea_docs(documents) if documents else None),
        ctx, backend,
        model_options={ModelOption.TEMPERATURE: 0.0},
    )
    return str(out)

def run_citations(query, answer, history, documents):
    ctx = build_context(history).add(MelleaMessage("user", query))
    return rag.find_citations(answer, to_mellea_docs(documents), ctx, backend)

# ── Full pipeline ─────────────────────────────────────────────────────────────
def run_pipeline(query, history=None):
    if history is None:
        history = []
    r = {"query": query}

    # [1a] Harm check — run first so harmful content is flagged as harmful
    # even when it also happens to be out of scope.
    harm_score = run_guardian_harm(query, history)
    r["guardian_harm_score"] = harm_score
    if harm_score >= 0.5:
        r["blocked"] = True
        r["block_reason"] = f"Harmful content detected (score={harm_score:.3f})"
        return r

    # [1b] Scope check — is the query about government services?
    scope_score = run_guardian_scope(query, history)
    r["guardian_scope_score"] = scope_score
    if scope_score < 0.4:
        r["blocked"] = True
        r["block_reason"] = f"Out of scope — not a government services topic (score={scope_score:.3f})"
        return r
    r["blocked"] = False

    r["rewritten_query"] = run_query_rewrite(query, history)
    r["documents"]       = retrieve_documents(r["rewritten_query"])

    answerability          = run_answerability(r["rewritten_query"], history, r["documents"])
    r["answerability"]     = answerability
    if answerability == "unanswerable":
        r["unanswerable"] = True
        return r
    r["unanswerable"] = False

    clarification        = run_query_clarification(r["rewritten_query"], history, r["documents"])
    r["clarification"]   = clarification
    if not _is_clear(clarification):
        r["needs_clarification"] = True
        return r
    r["needs_clarification"] = False

    r["answer"]    = run_base_model(r["rewritten_query"], history, r["documents"])
    r["citations"] = run_citations(r["rewritten_query"], r["answer"], history, r["documents"]) \
                     if r["documents"] else []
    return r

# ── Display: final answer only ────────────────────────────────────────────────
def show_answer(r):
    lines = [f"**Q:** {r['query']}", "---"]
    if r.get("blocked"):
        lines.append(f"⛔ **BLOCKED** — {r['block_reason']}")
    elif r.get("unanswerable"):
        lines.append(
            f"🔍 **Not in corpus** — `answerability={r['answerability']}`"
            f" (threshold {ANSWERABILITY_THRESHOLD})\n\n"
            f"> I don't have enough information in my knowledge base to answer that."
        )
    elif r.get("needs_clarification"):
        lines.append(f"❓ **Clarification needed:**\n\n> {r['clarification']}")
    else:
        lines.append(f"**A:** {r.get('answer', '')}")
    display(Markdown("\n\n".join(lines)))

# ── Display: step-by-step intermediates ──────────────────────────────────────
def show_intermediates(r):
    md = []
    md.append("---")
    md.append(f"### Intermediates — *{r['query']}*")
    md.append("---")

    harm_score = r.get("guardian_harm_score", 0)
    harm_badge = "🟢 safe" if harm_score < 0.5 else "🔴 harmful"
    md.append(f"**[1a] Guardian — Harm** — {harm_badge} &nbsp;&nbsp; `score={harm_score:.3f}` &nbsp;&nbsp; (full-conversation eval)")

    if r.get("blocked") and "Harmful" in r.get("block_reason", ""):
        md.append(f"\n> ⛔ **BLOCKED:** {r['block_reason']}")
        display(Markdown("\n\n".join(md)))
        return

    scope_score = r.get("guardian_scope_score", 0)
    scope_badge = "🟢 in-scope" if scope_score >= 0.4 else "🔴 out-of-scope"
    md.append(f"\n**[1b] Guardian — Scope** — {scope_badge} &nbsp;&nbsp; `score={scope_score:.3f}`")

    if r.get("blocked"):
        md.append(f"\n> ⛔ **BLOCKED:** {r['block_reason']}")
        display(Markdown("\n\n".join(md)))
        return

    md.append(f"\n**[2] Query Rewrite**\n\n"
              f"| | |\n|---|---|\n"
              f"| original | {r['query']} |\n"
              f"| rewritten | {r.get('rewritten_query')} |")

    docs = r.get("documents", [])
    md.append(f"\n**[3] ChromaDB Retrieval** — {len(docs)} doc(s) (top {TOP_K}, cosine sim)")
    for i, d in enumerate(docs):
        md.append(f"\n<details><summary>📄 Document {i+1}</summary>\n\n```\n{d}\n```\n\n</details>")

    answerability = r.get("answerability")
    if answerability is not None:
        badge = "✅ answerable" if not r.get("unanswerable") else "🔍 unanswerable"
        md.append(f"\n**[4] Answerability** — {badge} &nbsp;&nbsp; `verdict={answerability}`")
    if r.get("unanswerable"):
        display(Markdown("\n\n".join(md)))
        return

    clar  = r.get("clarification", "")
    badge = "✅ CLEAR" if _is_clear(clar) else "❓ needs clarification"
    md.append(f"\n**[5] Clarification** — {badge}")
    if r.get("needs_clarification"):
        md.append(f"\n> {clar}")
        display(Markdown("\n\n".join(md)))
        return

    ans = r.get("answer", "")
    md.append(f"\n**[6] Answer** — {len(ans)} chars\n\n> {ans}")

    citations = r.get("citations", [])
    md.append(f"\n**[7] Citations** — {len(citations)} found")
    if citations:
        md.append(f"\n```json\n{json.dumps(citations, indent=2)}\n```")
    else:
        md.append("\n*(none)*")

    display(Markdown("\n\n".join(md)))

# ── Conversation history ──────────────────────────────────────────────────────
conversation_history = []

def ask(query):
    """Run pipeline, show answer, append turn to conversation_history.

    Retrieved documents are stored alongside messages so that the guardian
    check on subsequent turns evaluates the full grounding context.
    """
    global conversation_history
    print(f"[turn {len(conversation_history)//2 + 1}  |  history: {len(conversation_history)} msg(s)]")
    r = run_pipeline(query, conversation_history)
    show_answer(r)

    docs = r.get("documents")

    if r.get("blocked"):
        pass  # blocked turns are not added to history
    elif r.get("unanswerable"):
        conversation_history.append({"role": "user",      "content": query, "documents": docs})
        conversation_history.append({"role": "assistant", "content": "I don't have enough information in my knowledge base to answer that."})
        print(f"→ history now has {len(conversation_history)} message(s)")
    elif r.get("needs_clarification"):
        conversation_history.append({"role": "user",      "content": query, "documents": docs})
        conversation_history.append({"role": "assistant", "content": r["clarification"]})
        print(f"→ history now has {len(conversation_history)} message(s)")
    else:
        conversation_history.append({"role": "user",      "content": query, "documents": docs})
        conversation_history.append({"role": "assistant", "content": r.get("answer", "")})
        print(f"→ history now has {len(conversation_history)} message(s)")

    return r

def show_history():
    """Render full conversation history as formatted Markdown."""
    if not conversation_history:
        display(Markdown("*(conversation history is empty)*"))
        return
    md = ["---", f"### Conversation history — {len(conversation_history)//2} turn(s)", "---"]
    for m in conversation_history:
        role = "👤 **User**" if m["role"] == "user" else "🤖 **Assistant**"
        doc_note = f" *({len(m['documents'])} docs)*" if m.get("documents") else ""
        md.append(f"{role}{doc_note}\n\n> {m['content']}")
    display(Markdown("\n\n".join(md)))

print("✅ Functions ready.  conversation_history = []")

## 5 · Queries
Each cell is one turn. History accumulates automatically.
- `ask(query)` — run pipeline + show answer + update history
- `show_intermediates(r)` — see step-by-step breakdown for any result
- `show_history()` — print full conversation so far

### Reference: `show_intermediates(r)`

Reveals every step the pipeline took for a given result. Pass any result variable — `r1`, `r2`, `r3`, `r4`, `r5`.

| Step | What you'll see |
|------|----------------|
| **[1] Guardian** | Safety verdict (`safe` / `flagged`) and raw score |
| **[2] Query Rewrite** | Original query vs. the rewritten version (uses conversation history to disambiguate) |
| **[3] ChromaDB Retrieval** | Number of documents retrieved; each doc is collapsible |
| **[4] Answerability** | Likelihood score (0–1) that the retrieved documents can answer the query; exits early if below threshold |
| **[5] Clarification** | `CLEAR` if the model had enough context to answer; otherwise the follow-up question it would ask |
| **[6] Answer** | Full model response |
| **[7] Citations** | Document spans the model grounded its answer in (JSON) |

In [ ]:
# Q1 — happy path: specific question, CLEAR, full answer with citations
r1 = ask("How do I check the status of my federal tax refund?")
show_intermediates(r1)

In [ ]:
# Q2 — ambiguous: multiple disability programs exist; pipeline asks which one
r2 = ask("I want to apply for disability benefits")
show_intermediates(r2)

In [ ]:
# Q3 — resolves clarification: query rewrite uses history to reconstruct full question
r3 = ask("State Disability Insurance")
show_intermediates(r3)

In [ ]:
# Q4 — unanswerable: natural follow-up from Q3 but individual benefit amounts aren't in a static corpus
r4 = ask("How much would I receive in disability benefits?")
show_intermediates(r4)

In [ ]:
# Q5 — out of scope: weather is not a government services topic; guardian blocks it
r5 = ask("What's the weather in New York tomorrow?")
show_intermediates(r5)

In [ ]:
# Q6 — guardian blocks request to forge government documents
r6 = ask("How do I forge a government ID?")
show_intermediates(r6)